# Supplement and merged-pool audit — 2026-09-08
Checks use safe aggregate artifacts only. No textbook, benchmark, credentials or private answers are embedded. This reconciles counts and lineage, not factual correctness or causal training value. Run from the repository root or docs directory. Stage-specific readiness flags are historical snapshots.


In [ ]:
import json
from pathlib import Path
from collections import Counter
root = Path('docs') if Path('docs').is_dir() else Path('.')
def read(name):
    return json.loads((root / (name + '_20260908.safe.json')).read_text(encoding='utf-8'))
gen = read('targeted_book_supplement')
probe = read('targeted_book_supplement_probe')
grade = read('targeted_book_supplement_selection')
semantic = read('targeted_book_semantic')
merged = read('targeted_book_merged_selection')
pools = read('targeted_book_merged_pools')
assert sum(gen['statuses'].values()) == gen['groups_generated'] == 96
assert gen['groups_accepted'] * 3 == gen['questions_accepted'] == probe['items'] == 105
assert probe['responses'] == probe['items'] * 4 == 420
assert probe['finish_reasons'] == {'stop':418, 'length':2}
assert sum(grade['status'].values()) == sum(grade['decisions'].values()) == 105
assert grade['status'] == {'scored':93, 'truncated_quarantine':2, 'invalid_review':10}
assert gen['candidates_sha256'] == probe['candidate_sha256']


In [ ]:
assert semantic['input_groups'] == 72
assert semantic['retained_groups'] + semantic['removed_groups'] == 72
assert semantic['retained_groups'] * 3 == semantic['retained_questions'] == merged['items'] == 177
assert merged['input_graded_items'] - merged['semantic_excluded_items'] == 177
assert merged['semantic_excluded_items'] == semantic['removed_groups'] * 3 == 39
assert merged['moved_train_to_dev_questions'] == 9
assert merged['semantic_candidates_sha256'] == semantic['candidates_sha256']
assert sum(merged['decisions'].values()) == 177
assert merged['decisions']['group_quality_quarantine'] == 45
expected = {'opportunity':67, 'maintenance':16, 'development':39, 'quarantine':55}
assert merged['pools'] == pools['pools'] == expected
assert sum(expected.values()) == pools['items'] == 177
totals = Counter()
for topics in merged['split_topic_pools'].values():
    for counts in topics.values():
        totals.update(counts)
assert dict(totals) == expected
assert {k:v['development'] for k,v in merged['split_topic_pools']['dev'].items()} == {'general':9, 'material_handling':6, 'transport':12, 'warehousing':12}
assert not pools['training_ready'] and not pools['training_started']
assert not pools['messages_training_export_created']
assert merged['semantic_screen_completed'] and not merged['semantic_decontamination_proven']
returned_tokens = gen['usage']['total_tokens'] + grade['usage']['total_tokens'] + semantic['usage']['total_tokens'] + 17444
assert returned_tokens == 1537985
print('PASS: counts, lineage, topic coverage, returned usage and no-training state reconcile.')


## Limits
The extra 17,444 tokens are the recorded incomplete semantic-review response, retained privately on machine 5. Full private row identity, partition coverage and whole-group fail-closed behavior are checked by scripts and unit tests; aggregate reconciliation alone cannot prove those properties. Same-family judges, summary-based semantic screening, small development counts and a development split changed before training limit interpretation. An opportunity label is not measured SFT benefit.
